In [0]:
-- Merge incremental transactions into the main Bronze transactions table
-- This ensures idempotent ingestion and avoids duplicate transaction_ids

CREATE TABLE IF NOT EXISTS coffee.bronze.transactions
USING DELTA
AS
SELECT * FROM coffee.bronze.transactions_batch
WHERE 1 = 0;



In [0]:
MERGE INTO coffee.bronze.transactions t
USING coffee.bronze.transactions_batch b
ON t.transaction_id = b.transaction_id

WHEN NOT MATCHED
AND b.transaction_id IS NOT NULL
THEN INSERT *
;



In [0]:
MERGE INTO coffee.bronze.transactions t
USING coffee.bronze.transactions_incremental i
ON t.transaction_id = i.transaction_id

WHEN MATCHED THEN
UPDATE SET
  t.store_id           = i.store_id,
  t.payment_method_id  = i.payment_method_id,
  t.voucher_id         = i.voucher_id,
  t.user_id            = i.user_id,
  t.original_amount    = i.original_amount,
  t.discount_applied   = i.discount_applied,
  t.final_amount       = i.final_amount,
  t.created_at         = i.created_at,

  -- refresh metadata
  t.loaded_at          = i.loaded_at,
  t.updated_at         = i.updated_at,
  t.load_dt            = i.load_dt,
  t.source             = i.source,
  t.source_file        = i.source_file

WHEN NOT MATCHED
AND i.transaction_id IS NOT NULL
THEN INSERT *;
